# Natural Language Processing
## Assignment 2 - Group 17

*_Adriano Machado (up202105352), Félix Martins (up202108837), Francisco da Ana (up202108762)_*  


## 1. Introduction

This assignment focuses on politeness classification using the Polite Guard dataset. We explore various techniques including fine-tuning transformer models, domain adaptation, parameter-efficient fine-tuning (PEFT), and prompting large language models (LLMs).


### 1.1. Data: Polite Guard Dataset

The [Polite Guard dataset](https://github.com/intel/polite-guard), developed by Intel, is used for **politeness classification** into four categories: polite, somewhat polite, neutral, or impolite.

**Key Features:**,

* Specialized corpus for politeness in text.
* **Sources:**
    * 100k synthetic samples generated by LLMs (50k Few-Shot & 50k using Chain-of-Thought prompting).
    * 200 real-world samples from corporate training (with personal identifiers removed).
* No personal identifiers are present in the dataset. 
* The dataset aims for an even distribution of classes (approximately 25% for each class in every split).

**Performance Benchmark (from official dataset webpage):**
* Accuracy: 92%
* F1-Score: 92%




### 1.2.  Categories

Four-category politeness classification:

1.  **Polite:** Courteous and respectful text with friendly tone.
2.  **Somewhat Polite:** Respectful but less warm or formal.
3.  **Neutral:** Factual, straightforward text without politeness attempts.
4.  **Impolite:** Rude, blunt, and disrespectful text.


### 1.3. Data Structure

Each example includes:

*   **text:** Input text string.
*   **label:** Politeness category (polite, somewhat polite, neutral, impolite).
*   **source:** Generation model ("LMS" for real data).
*   **reasoning:** LLM rationale (for synthetic data with Chain-of-Thought prompting).

## 2.0 Data Exploration

The code for this section was already provided in the first assignment notebook. Here we will just show the results of the data exploration.

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Dataset</th>
      <th>Rows</th>
      <th>Duplicates</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>train_cot</td>
      <td>40000</td>
      <td>0</td>
    </tr>
    <tr>
      <th>1</th>
      <td>train_few_shot</td>
      <td>40000</td>
      <td>0</td>
    </tr>
    <tr>
      <th>2</th>
      <td>val_cot</td>
      <td>5000</td>
      <td>0</td>
    </tr>
    <tr>
      <th>3</th>
      <td>val_few_shot</td>
      <td>5000</td>
      <td>0</td>
    </tr>
    <tr>
      <th>4</th>
      <td>test_cot</td>
      <td>5000</td>
      <td>0</td>
    </tr>
    <tr>
      <th>5</th>
      <td>test_few_shot</td>
      <td>5000</td>
      <td>0</td>
    </tr>
    <tr>
      <th>6</th>
      <td>test_LMs</td>
      <td>200</td>
      <td>0</td>
    </tr>
  </tbody>
</table>
</div>


There's no duplicates and no missing values in our datasets.

There's about 80000 (80%) instances of training, 10000 (10%) instances of validation and 10000 (10%) instances of test (plus 200 real world examples)

<img src="../assign1/docs/EDA/politeness_count.png" alt="Politeness Count">

The synthetic datasets show similar distributions across politeness labels. In contrast, the real dataset is slightly biased, with a higher proportion of "polite" and "neutral" labels compared to "impolite" and "somewhat polite" ones.

<img src="../assign1/docs/EDA/politeness_boxplot.png" alt="Politeness Boxplot">

The synthetic datasets exhibit a tendency where longer texts are associated with higher politeness levels. This pattern, however, is not observed in the real test set, indicating potential biases or artificial correlations inherent in the synthetic data generation.

<img src="../assign1/docs/EDA/word_cloud.png" alt="Word Cloud">

Impolite language conveys frustration through words like "seriously" and "ridiculous." Neutral language adopts a factual and instructive tone, using terms like "please" and "note." Somewhat polite language attempts to mitigate negativity with words of apology and empathy, such as "sorry" and "hear." Polite language is marked by expressions of helpfulness and respect, including words like "happy" and "assistance."

# 3. 1st Assignment Review

In the first assignment, traditional machine learning models were explored for politeness classification.

## Models and Performance Summary

- **Models Tested:** Naive Bayes, Logistic Regression, and SVM
- **Feature Extraction:** Bag-of-Words (BoW_1, BoW_2), TF-IDF, and Word2Vec
- **Best Performance:** SVM + Word2Vec achieved **88.48% accuracy** and **88.5% F1-Score**
- **Key Findings:** The model showed balanced performance across politeness levels, with notably better performance for "lower" politeness labels (e.g., "impolite")

## Results Visualization

<img src="../assign1/results/plots/val_train_by_feature_model.png" alt="Validation Accuracy Comparison" width="700">

*Figure 3.1: Validation accuracy comparison across different models and feature extraction techniques from the 1st assignment.*

<img src="../assign1/results/confusion_matrix.png" alt="Confusion Matrix" width="600">

*Figure 3.2: Confusion matrix for the best-performing SVM + Word2Vec model on the test set.*

## Key Insights from Traditional ML Approach

The traditional machine learning approach established a strong baseline, with SVM demonstrating superior performance when combined with Word2Vec embeddings. The results highlight the effectiveness of semantic feature representations over simple count-based methods for politeness detection tasks.

## 4. Methodology: Transformer-Based Classification
For this assignment, we leverage transformer models for politeness classification. We explore full fine-tuning, domain adaptation, and parameter-efficient fine-tuning techniques.


### 4.1. Model Choices and Fine-Tuning
We selected two baseline transformer models:
* **(1) BERT (bert-base-uncased):**
    * Its encoder-only architecture is well-suited for understanding nuanced phrases and context. 
    * Pre-trained on large, diverse English corpora, aligning with the dataset's domain and tone. 
    * Widely used in NLP with good performance in sequence classification. 
* **(2) RoBERTa (roberta-base):**
    * An improvement over BERT. 
    * Not uncased, potentially capturing more information. 

In [2]:
import os
import json
import logging 
from dataclasses import dataclass, field
from datetime import datetime
from typing import Dict, List

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

import wandb
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    AutoModelForMaskedLM,                
    DataCollatorForLanguageModeling,     
    set_seed                             
)
from transformers.integrations import WandbCallback 
import torch                                 
import torch.nn.functional as F              

try:
    from kaggle_secrets import UserSecretsClient
except ImportError:
    UserSecretsClient = None
    print("Kaggle UserSecretsClient not found. Environment variables should be set manually if not on Kaggle.")


2025-05-24 18:01:38.896476: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748106099.039092   22400 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748106099.090087   22400 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1748106099.408594   22400 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748106099.408755   22400 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748106099.408767   22400 computation_placer.cc:177] computation placer alr

Kaggle UserSecretsClient not found. Environment variables should be set manually if not on Kaggle.



#### 4.1.1. Hyperparameter Tuning
Hyperparameters such as learning rate and weight decay were tuned for both models. 
The best configurations were selected based on the F1-score on the validation set. 
Initially, models were tuned for a few epochs (e.g., 1-3 based on the sweep config) and then the best models were trained for 10 epochs. 

**Hyperparameter Sweep Configuration (example):**
* `learning_rate`: e.g., [1e-6, 1e-5, 2e-5]
* `batch_size`: e.g., [64]
* `weight_decay`: e.g., [0.0, 0.01, 0.1]
* `num_train_epochs`: e.g., [1] (for initial sweep trials)


### Sweep and Base Configuration
These dataclasses define the parameters for the hyperparameter sweep and the base configuration for the models.



In [3]:
@dataclass
class SweepConfig:
    method: str = "grid"
    metric: Dict[str, str] = field(default_factory=lambda: {"name": "eval_f1", "goal": "maximize"})
    parameters: Dict[str, Dict] = field(default_factory=lambda: {
        "learning_rate": {"values": [1e-6, 1e-5, 2e-5]},
        "batch_size": {"values": [64]},
        "weight_decay": {"values": [0.0, 0.01, 0.1]},
        "num_train_epochs": {"values": [1]}, 
    })
    max_trials: int = 9 

@dataclass
class BaseConfig:
    model_name: str = "roberta-base" 
    dataset_name: str = "Intel/polite-guard" 
    wandb_project: str = "polite-guard-tuning" 

base_cfg_sweep = BaseConfig()
sweep_cfg_global = SweepConfig()

### Setup Secrets
This function handles API keys.

In [4]:
def setup_secrets_sweep():
    if UserSecretsClient:
        try:
            secrets = UserSecretsClient()
            os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
            hf_token = secrets.get_secret("huggingface")
            os.environ["HUGGINGFACE_TOKEN"] = hf_token
            print("Kaggle secrets loaded for sweep.")
        except Exception as e:
            print(f"Kaggle secrets not found for sweep: {e}. Ensure WANDB_API_KEY and HUGGINGFACE_TOKEN are set in your environment.")
    else:
        print("UserSecretsClient not available. Ensure WANDB_API_KEY and HUGGINGFACE_TOKEN are set in your environment if not on Kaggle.")

    if not os.environ.get("WANDB_API_KEY"):
        print("Warning: WANDB_API_KEY not set for sweep.")
    if not os.environ.get("HUGGINGFACE_TOKEN"):
        print("Warning: HUGGINGFACE_TOKEN not set for sweep.")

### Data Loading & Preprocessing for Hyperparameter Tuning
This function loads and tokenizes the dataset specifically for the hyperparameter tuning phase.



In [5]:
def get_dataset_info_sweep(dataset_name: str): 
    raw_train = load_dataset(dataset_name, split="train")
    label_list = sorted(raw_train.unique("label")) 
    label2id = {lbl: i for i, lbl in enumerate(label_list)}
    return label_list, label2id

def load_and_preprocess_sweep(cfg: BaseConfig, tokenizer_for_preprocessing: AutoTokenizer, label2id_map: Dict[str, int]): 
    print(f"Loading dataset {cfg.dataset_name} for sweep")
    ds = DatasetDict({
        "train": load_dataset(cfg.dataset_name, split="train"),
        "validation": load_dataset(cfg.dataset_name, split="validation")
    })

    def preprocess_sweep(batch):
        toks = tokenizer_for_preprocessing(batch["text"], truncation=True, padding="max_length", max_length=tokenizer_for_preprocessing.model_max_length)
        toks["labels"] = [label2id_map[l] for l in batch["label"]]
        return toks

    tokenized = ds.map(
        preprocess_sweep,
        batched=True,
        remove_columns=ds["train"].column_names,
    )
    print("Dataset preprocessed for sweep.")
    return tokenized

### Model Builder & Metrics for Hyperparameter Tuning
This function sets up the model and trainer for each sweep trial.

In [6]:
def build_trainer_sweep(
    cfg: BaseConfig,
    tokenized: DatasetDict,
    tokenizer: AutoTokenizer,
    num_labels: int,
    learning_rate: float,
    weight_decay: float,
    num_train_epochs: int,
    batch_size: int
) -> Trainer:
    print(f"Building model {cfg.model_name} with lr={learning_rate}, wd={weight_decay}, epochs={num_train_epochs}, bs={batch_size} for sweep")
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name, num_labels=num_labels
    )

    def compute_metrics_sweep(eval_pred): 
        preds = np.argmax(eval_pred.predictions, axis=-1)
        labels = eval_pred.label_ids
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, preds, average="weighted", zero_division=0
        )
        acc = accuracy_score(labels, preds)
        return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

    current_run_id = wandb.run.id if wandb.run else datetime.now().strftime("%Y%m%d-%H%M%S")
    output_dir_sweep = f"./sweep_output/{current_run_id}"
    logging_dir_sweep = f"./logs_sweep/{current_run_id}" 

    training_args = TrainingArguments(
        output_dir=output_dir_sweep,
        eval_strategy="epoch", 
        save_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_train_epochs,
        weight_decay=weight_decay,
        logging_dir=logging_dir_sweep,
        report_to="wandb",
        fp16=torch.cuda.is_available(), 
        load_best_model_at_end=True,
        metric_for_best_model="f1", 
        greater_is_better=True,
        logging_steps=10, 
        save_total_limit=1 
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        data_collator=data_collator,
        compute_metrics=compute_metrics_sweep,
        tokenizer=tokenizer 
    )
    return trainer

### Sweep Orchestration
This is the main function to set up and run the W&B sweep.

In [7]:
def sweep_orchestration(): 
    setup_secrets_sweep()

    sweep_id = wandb.sweep(
        sweep={
            "method": sweep_cfg_global.method,
            "metric": sweep_cfg_global.metric,
            "parameters": sweep_cfg_global.parameters,
        },
        project=base_cfg_sweep.wandb_project,
    )

    def sweep_train_trial():
        run = wandb.init()  

        lr = wandb.config.learning_rate
        bs = wandb.config.batch_size 
        wd = wandb.config.weight_decay
        ne = wandb.config.num_train_epochs

        current_model_name = base_cfg_sweep.model_name

        run_name = f"{current_model_name.replace('/', '-')}_lr{lr}_wd{wd}_ep{ne}_bs{bs}"
        wandb.run.name = run_name

        print(f"Starting W&B sweep trial: {run_name} with ID: {wandb.run.id}")
        print(f"Hyperparameters: lr={lr}, bs={bs}, wd={wd}, ne={ne}, model={current_model_name}")

        _, label2id_sweep = get_dataset_info_sweep(base_cfg_sweep.dataset_name)
        num_labels_sweep = len(label2id_sweep)

        tokenizer_sweep = AutoTokenizer.from_pretrained(current_model_name)
        tokenized_datasets_sweep = load_and_preprocess_sweep(base_cfg_sweep, tokenizer_sweep, label2id_sweep)

        trainer_sweep = build_trainer_sweep(
            base_cfg_sweep,
            tokenized_datasets_sweep,
            tokenizer_sweep,
            num_labels_sweep,
            learning_rate=lr,
            weight_decay=wd,
            num_train_epochs=ne,
            batch_size=bs, 
        )
        print("Starting trainer_sweep.train()")
        trainer_sweep.train()
        print("Training finished. Starting evaluation...")
        eval_metrics = trainer_sweep.evaluate()
        wandb.log(eval_metrics) 

        print(f"Sweep trial {run_name} finished. Metrics: {eval_metrics}")

    wandb.agent(sweep_id, function=sweep_train_trial, count=sweep_cfg_global.max_trials)
    print("W&B sweep agent finished.")

# To run the sweep in a notebook cell:
# sweep_orchestration()

Hyperparameters such as learning rate and weight decay were tuned for both models. 
The best configurations were selected based on the F1-score on the validation set. 
Initially, models were tuned for a few epochs (e.g., 1-3 based on the sweep config) and then the best models were trained for 10 epochs. 

**Hyperparameter Sweep Configuration (example):**
The `transformers_hyperparameter.py` script was used for this process. Key parameters explored include:
* `learning_rate`: e.g., [1e-6, 1e-5, 2e-5] 
* `batch_size`: e.g., [64]
* `weight_decay`: e.g., [0.0, 0.01, 0.1] 
* `num_train_epochs`: e.g., [1] (for initial sweep trials) 

**Selected Hyperparameters after Tuning (for 10 epochs training):**
* **BERT (bert-base-uncased):** Learning Rate = 2e-5, Weight Decay = 0.01 
* **RoBERTa (roberta-base):** Learning Rate = 2e-5, Weight Decay = 0.1 

(The results will be discussed in the results section.)



#### 4.2 Main Classification Pipeline and Domain Adaptation (MLM)
This section details the main training pipeline, incorporating the option for domain adaptation using Masked Language Modeling (MLM). The code is primarily from transformers_classification.py.

## Configuration for Main Pipeline
This dataclass defines the parameters for the main classification training runs, including settings for domain adaptation.

In [ ]:
@dataclass
class Config:
    model_name: str = "roberta-base"
    dataset_name: str = "Intel/polite-guard" 
    cl_epochs: int = 5 
    batch_size: int = 64 
    learning_rate: float = 2e-5 
    weight_decay: float = 0.1 
    wandb_project: str = "domain-adaptation-seeded" 
    mlm_epochs: int = 2 
    mlm_learning_rate: float = 2e-5 
    mlm_probability: float = 0.1 
    enable_domain_adapt: bool = True 
    mlm_validation_split: float = 0.1 
    seed: int = 42 

    # these will be set in __post_init__:
    run_name: str = None 
    output_dir: str = None 
    mlm_output_dir: str = None 

    def __post_init__(self): 
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S") 
        da_status = "DA" if self.enable_domain_adapt else "NoDA" 
        self.run_name = ( 
            f"{timestamp}_{self.model_name.replace('/', '-')}_{da_status}_"
            f"lr{self.learning_rate}_bs{self.batch_size}_ep{self.cl_epochs}_seed{self.seed}"
        )
        base_output_path = "./results"
        self.output_dir = os.path.join(base_output_path, "classification", self.run_name) 
        if self.enable_domain_adapt: 
            self.mlm_output_dir = os.path.join(base_output_path, "mlm", self.run_name) 
        else: 
            self.mlm_output_dir = None

## Setup Secrets (for Main Pipeline)
This function handles API keys. It's important to ensure that secrets like API keys are handled securely and are not hardcoded.


In [ ]:
try:
    from kaggle_secrets import UserSecretsClient 
except ImportError:
    UserSecretsClient = None 

def setup_secrets_main():
    if UserSecretsClient: 
        try:
            secrets = UserSecretsClient() 
            os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY") 
            hf_token = secrets.get_secret("huggingface") 
            os.environ["HUGGINGFACE_TOKEN"] = hf_token 
            print("Kaggle secrets loaded for main pipeline.")
        except Exception as e: 
            print(f"Kaggle secrets not found for main pipeline: {e}. Ensure WANDB_API_KEY and HUGGINGFACE_TOKEN are set.") 
    else: 
        print("UserSecretsClient not available. Ensure WANDB_API_KEY and HUGGINGFACE_TOKEN are set if not on Kaggle.")

    if not os.environ.get("WANDB_API_KEY"): 
        print("Warning: WANDB_API_KEY not set for main pipeline.") 
    if not os.environ.get("HUGGINGFACE_TOKEN"): 
        print("Warning: HUGGINGFACE_TOKEN not set for main pipeline.") 

### Data Loading & Preprocessing (Main Pipeline & MLM Data Prep)
This function loads the dataset and tokenizes it for both the main classification task and, if enabled, for Masked Language Modeling (MLM) pre-training.

In [ ]:
def load_and_preprocess_main(cfg: Config, tokenizer: AutoTokenizer): 
    print(f"Loading dataset {cfg.dataset_name} for main pipeline") 
    raw_datasets = load_dataset(cfg.dataset_name) 

    if "train" not in raw_datasets or "label" not in raw_datasets["train"].features:
        raise ValueError("Dataset must contain a 'train' split with a 'label' feature.")

    if hasattr(raw_datasets["train"].features["label"], "names"):
        label_list_train = raw_datasets["train"].features["label"].names 
        label2id = {name: i for i, name in enumerate(label_list_train)} 
    else:
        print("Label column is not ClassLabel. Creating mapping from unique values in 'train' split.")
        label_list_train = sorted(raw_datasets["train"].unique("label")) 
        if not label_list_train:
            raise ValueError("Could not determine labels from the 'train' split.")
        label2id = {lbl: i for i, lbl in enumerate(label_list_train)} 
    
    print(f"Labels found: {label_list_train}") 
    print(f"label2id mapping: {label2id}") 

    def tokenize_classification_main(batch): 
        outputs = tokenizer( 
            batch["text"], 
            truncation=True, 
            padding="max_length",
            max_length=tokenizer.model_max_length if tokenizer.model_max_length <= 512 else 512, 
        )
        outputs["labels"] = [label2id[l] for l in batch["label"]] 
        return outputs 

    print("Tokenizing labeled data for classification...") 
    columns_to_remove_for_classification = [col for col in raw_datasets["train"].column_names if col not in ["text", "label"]]

    tok_labeled = raw_datasets.map( 
        tokenize_classification_main, 
        batched=True, 
        remove_columns=raw_datasets["train"].column_names 
    )

    tok_unlabeled_train = None 
    tok_unlabeled_val = None 
    if cfg.enable_domain_adapt: 
        print("Tokenizing unlabeled data for MLM...") 
        mlm_dataset_source = raw_datasets["train"] 

        def tokenize_mlm_main(batch): 
            return tokenizer( 
                batch["text"], 
                truncation=True, 
                padding="max_length",
                max_length=tokenizer.model_max_length if tokenizer.model_max_length <= 512 else 512, 
            )

        if 0 < cfg.mlm_validation_split < 1: 
            mlm_splits = mlm_dataset_source.train_test_split( 
                test_size=cfg.mlm_validation_split, 
                seed=cfg.seed, 
            )
            tok_unlabeled_train = mlm_splits["train"].map( 
                tokenize_mlm_main, 
                batched=True, 
                remove_columns=mlm_splits["train"].column_names, 
            )
            tok_unlabeled_val = mlm_splits["test"].map( 
                tokenize_mlm_main, 
                batched=True, 
                remove_columns=mlm_splits["test"].column_names, 
            )
            print(f"MLM training data: {len(tok_unlabeled_train)} samples, MLM validation data: {len(tok_unlabeled_val)} samples") 
        else: 
            tok_unlabeled_train = mlm_dataset_source.map( 
                tokenize_mlm_main, 
                batched=True, 
                remove_columns=mlm_dataset_source.column_names, 
            )
            print(f"MLM training data: {len(tok_unlabeled_train)} samples (no MLM validation split)") 
    
    tok_unlabeled_data = {"train": tok_unlabeled_train, "validation": tok_unlabeled_val} if cfg.enable_domain_adapt else None 

    return tok_labeled, tok_unlabeled_data, tokenizer, label2id 

### Model Building & Metrics Definition (Main Pipeline)
These functions are responsible for loading the sequence classification model and defining the metrics that will be used for evaluation during training and testing.

In [ ]:
def build_model_and_metrics_main(model_path: str, num_labels: int, tokenizer_main: AutoTokenizer, label2id_map: Dict, id2label_map: Dict): 
    print(f"Loading classification model from {model_path}") 
    model = AutoModelForSequenceClassification.from_pretrained( 
        model_path, 
        num_labels=num_labels, 
        label2id=label2id_map,
        id2label=id2label_map,
        ignore_mismatched_sizes=True 
    )
    print(f"Successfully loaded model. Type: {type(model)}") 
    if model.get_input_embeddings().weight.shape[0] != len(tokenizer_main): 
        model.resize_token_embeddings(len(tokenizer_main)) 
        print(f"Resized model token embeddings to: {len(tokenizer_main)}")


    def compute_metrics_main(eval_pred): 
        logits, labels = eval_pred 
        preds = np.argmax(logits, axis=-1) 

        acc = accuracy_score(labels, preds) 
        precision, recall, f1, _ = precision_recall_fscore_support( 
            labels, preds, average="weighted", zero_division=0 
        )
        return { 
            "accuracy": acc, 
            "precision": precision, 
            "recall": recall, 
            "f1": f1, 
        }
    return model, compute_metrics_main 

def plot_and_save_confusion_matrix_main(cm: np.ndarray, labels_list: List[str], path: str): 
    plt.figure(figsize=(max(6, len(labels_list)), max(6, len(labels_list)))) 
    plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues) 
    plt.title("Confusion Matrix") 
    plt.colorbar()
    plt.ylabel("True Label") 
    plt.xlabel("Predicted Label") 

    tick_marks = np.arange(len(labels_list)) 
    plt.xticks(tick_marks, labels_list, rotation=45, ha="right") 
    plt.yticks(tick_marks, labels_list) 

    thresh = cm.max() / 2.0 
    for i, j in np.ndindex(cm.shape): 
        plt.text( 
            j, 
            i, 
            f"{cm[i, j]}", 
            ha="center", 
            va="center", 
            color="white" if cm[i, j] > thresh else "black", 
        )
    plt.tight_layout() 
    os.makedirs(os.path.dirname(path), exist_ok=True) 
    plt.savefig(path) 
    print(f"Confusion matrix saved to {path}")
    plt.close() 

### Domain Adaptation (MLM) Function
To adapt the pre-trained models to the specific language of the Polite Guard dataset, we performed intermediate fine-tuning using Masked Language Modeling (MLM) on the target corpus (training set texts). The perform_domain_adaptation_main function, detailed below, handles this by training an AutoModelForMaskedLM. The effectiveness of this phase can be tracked by observing the perplexity on a held-out portion of the MLM training data.

In [ ]:

def perform_domain_adaptation_main(cfg: Config, tok_unlabeled_data: Dict[str, DatasetDict], tokenizer_mlm: AutoTokenizer): 
    print("Starting domain adaptation (MLM) training") 
    
    mlm_model = AutoModelForMaskedLM.from_pretrained(cfg.model_name) 
    if mlm_model.getinput_embeddings().weight.shape[0] != len(tokenizer_mlm): 
        mlm_model.resize_token_embeddings(len(tokenizer_mlm)) 
    
    mlm_data_collator = DataCollatorForLanguageModeling( 
        tokenizer=tokenizer_mlm, 
        mlm=True, 
        mlm_probability=cfg.mlm_probability 
    )
    mlm_training_args = TrainingArguments( 
        output_dir=cfg.mlm_output_dir, 
        overwrite_output_dir=True, 
        num_train_epochs=cfg.mlm_epochs, 
        per_device_train_batch_size=cfg.batch_size, 
        per_device_eval_batch_size=cfg.batch_size, 
        save_steps=10000, 
        save_total_limit=2, 
        logging_steps=50 if not tok_unlabeled_data.get("validation") else max(1, int(len(tok_unlabeled_data["train"]) / (cfg.batch_size * 10))), # Dynamic logging steps [cite: 1]
        learning_rate=cfg.mlm_learning_rate, 
        weight_decay=cfg.weight_decay, 
        report_to="wandb", 
        load_best_model_at_end=(tok_unlabeled_data.get("validation") is not None), 
        evaluation_strategy="epoch" if tok_unlabeled_data.get("validation") else "no", 
        save_strategy="epoch" if tok_unlabeled_data.get("validation") else "steps", 
        seed=cfg.seed, 
        data_seed=cfg.seed, 
        fp16=torch.cuda.is_available(), 
    )
    
    mlm_trainer_kwargs = { 
        "model": mlm_model, 
        "args": mlm_training_args, 
        "train_dataset": tok_unlabeled_data["train"], 
        "data_collator": mlm_data_collator, 
    }
    
    if tok_unlabeled_data.get("validation"): 
        mlm_trainer_kwargs["eval_dataset"] = tok_unlabeled_data["validation"] 
            
    mlm_trainer = Trainer(**mlm_trainer_kwargs) 
    
    print("Starting MLM training") 
    mlm_trainer.train() 
    print("MLM training finished.") 
    
    if tok_unlabeled_data.get("validation"): 
        try:
            print("Evaluating MLM model for perplexity...") 
            metrics = mlm_trainer.evaluate(eval_dataset=tok_unlabeled_data["validation"]) 
            eval_loss = metrics.get("eval_loss") 
            if eval_loss is not None: 
                final_perplexity = np.exp(eval_loss) 
                wandb.log({"final_mlm_perplexity": final_perplexity, "mlm_eval_loss": eval_loss}) 
                print(f"Final MLM perplexity on MLM validation set: {final_perplexity:.4f} (from eval_loss: {eval_loss:.4f})") 
            else: 
                print("Could not find 'eval_loss' in MLM trainer evaluation metrics.") 
        except Exception as e: 
            print(f"Could not compute perplexity from trainer.evaluate(): {e}") 
    
    print(f"Saving domain-adapted MLM model to {cfg.mlm_output_dir}") 
    mlm_trainer.save_model(cfg.mlm_output_dir) 
    tokenizer_mlm.save_pretrained(cfg.mlm_output_dir) 
    print(f"Domain adaptation complete. Model saved to {cfg.mlm_output_dir}") 
    
    return cfg.mlm_output_dir 

### Main Training Pipeline Function
This function orchestrates the entire process: setting up configurations, loading and preprocessing data, optionally performing domain adaptation via MLM, training the sequence classification model, evaluating it on the test set, and logging results.

In [ ]:
def main_pipeline(): 
    cfg = Config() 
    setup_secrets_main() 

    set_seed(cfg.seed) 
    torch.manual_seed(cfg.seed) 
    if torch.cuda.is_available(): 
        torch.cuda.manual_seed_all(cfg.seed) 
    np.random.seed(cfg.seed)
    print(f"Global seed set to {cfg.seed}") 

    wandb.init( 
        project=cfg.wandb_project, 
        name=cfg.run_name, 
        config=vars(cfg), 
    )

    tokenizer_main_pipeline = AutoTokenizer.from_pretrained(cfg.model_name) 
    tok_labeled_data, tok_unlabeled_data, _, label2id_main = load_and_preprocess_main(cfg, tokenizer_main_pipeline) 
    
    num_labels_main = len(label2id_main) 
    id2label_main = {v: k for k,v in label2id_main.items()} 

    adapted_model_path = cfg.model_name 
    if cfg.enable_domain_adapt and tok_unlabeled_data and tok_unlabeled_data.get("train"): 
        print("--- Starting Domain Adaptation Phase ---") 
        adapted_model_path = perform_domain_adaptation_main(cfg, tok_unlabeled_data, tokenizer_main_pipeline) 
        print(f"Domain adaptation finished. Using adapted model from: {adapted_model_path}") 
    else: 
        print("Skipping domain adaptation or no unlabeled data provided for MLM.") 

    print("\n--- Starting Text Classification Phase ---") 
    print(f"Building classification model using weights from: {adapted_model_path}") 
    cl_model, compute_metrics_fn_main = build_model_and_metrics_main( 
        adapted_model_path, num_labels_main, tokenizer_main_pipeline, label2id_main, id2label_main 
    )

    cl_training_args = TrainingArguments( 
        output_dir=cfg.output_dir, 
        evaluation_strategy="epoch", 
        save_strategy="epoch", 
        learning_rate=cfg.learning_rate, 
        per_device_train_batch_size=cfg.batch_size, 
        per_device_eval_batch_size=cfg.batch_size * 2, 
        num_train_epochs=cfg.cl_epochs, 
        weight_decay=cfg.weight_decay, 
        load_best_model_at_end=True, 
        metric_for_best_model="f1", 
        greater_is_better=True, 
        report_to="wandb", 
        seed=cfg.seed, 
        data_seed=cfg.seed, 
        fp16=torch.cuda.is_available(), 
        logging_steps=max(1, int(len(tok_labeled_data["train"]) / (cfg.batch_size * 10))), 
        save_total_limit=2, 
    )
    
    cl_data_collator = DataCollatorWithPadding(tokenizer=tokenizer_main_pipeline) 
    
    cl_trainer = Trainer( 
        model=cl_model, 
        args=cl_training_args, 
        train_dataset=tok_labeled_data["train"], 
        eval_dataset=tok_labeled_data["validation"], 
        data_collator=cl_data_collator, 
        compute_metrics=compute_metrics_fn_main, 
        tokenizer=tokenizer_main_pipeline, 
    )
    print("Starting classification training") 
    cl_trainer.train() 
    print("Classification training finished.") 

    print("\n--- Evaluating on Test Set ---") 
    if "test" not in tok_labeled_data: 
        print("Warning: Test set not found in tokenized data. Skipping test evaluation.") 
    else: 
        pred_output = cl_trainer.predict(tok_labeled_data["test"]) 
        test_metrics = pred_output.metrics 
        print(f"Test Metrics: {test_metrics}") 
        wandb.log({f"test_{k}": v for k, v in test_metrics.items()}) 

        y_true_test = pred_output.label_ids 
        y_pred_test_logits = pred_output.predictions 
        y_pred_test_labels = np.argmax(y_pred_test_logits, axis=-1) 
        
        cm = confusion_matrix(y_true_test, y_pred_test_labels) 
        cm_path = os.path.join(cfg.output_dir, "confusion_matrix_test.png") 
        plot_and_save_confusion_matrix_main(cm, list(id2label_main.values()), cm_path) 
        wandb.log({"test_confusion_matrix": wandb.Image(cm_path)}) 

        print("Saving final classification model and results...") 
        cl_trainer.save_model(cfg.output_dir) 
        
        results_summary = { 
            "test_metrics": test_metrics, 
            "confusion_matrix_values": cm.tolist(), 
            "label2id": label2id_main, 
            "id2label": id2label_main, 
            "config_used": vars(cfg) 
        }
        results_path = os.path.join(cfg.output_dir, "test_results_summary.json") 
        os.makedirs(cfg.output_dir, exist_ok=True) 
        with open(results_path, "w") as f: 
            json.dump(results_summary, f, indent=4) 
        print(f"Results summary saved to {results_path}") 
        
        try: 
            raw_test_texts_dataset = load_dataset(cfg.dataset_name, split="test") 
            raw_test_texts = raw_test_texts_dataset["text"] 

            misclassification_table = wandb.Table(columns=["text_id", "text", "true_label", "predicted_label", "logits"]) 
            num_misclassified_to_log = min(100, len(raw_test_texts))

            misclassified_count = 0
            for i in range(len(y_true_test)): 
                if i < len(raw_test_texts): 
                    true_label_id = y_true_test[i] 
                    pred_label_id = y_pred_test_labels[i] 
                    if true_label_id != pred_label_id: 
                        if misclassified_count < num_misclassified_to_log:
                            misclassification_table.add_data( 
                                i, 
                                raw_test_texts[i], 
                                id2label_main[true_label_id], 
                                id2label_main[pred_label_id], 
                                y_pred_test_logits[i].tolist() 
                            )
                            misclassified_count +=1
                        else:
                            break 
            if misclassified_count > 0:
                 wandb.log({"test_misclassifications_sample": misclassification_table}) 
            else:
                print("No misclassifications found in the test set or logging limit is 0.")
        except Exception as e: 
            print(f"Could not log misclassifications: {e}") 

        try: 
            y_pred_test_probas = F.softmax(torch.tensor(y_pred_test_logits), dim=-1).numpy() 
            wandb.log({ 
                "test_roc_curve": wandb.plot.roc_curve(y_true_test, y_pred_test_probas, labels=list(id2label_main.values())), 
                "test_pr_curve": wandb.plot.pr_curve(y_true_test, y_pred_test_probas, labels=list(id2label_main.values())), 
            })
        except Exception as e: 
            print(f"Could not log multiclass ROC/PR curves: {e}") 
            print("This might happen if a class has no true samples, if wandb version is old, or if y_true/y_pred are not suitable.") 

    wandb.finish() 
    print("\n--- Main pipeline finished. Results saved and logged to W&B. ---") 

In [ ]:
# To run the main classification pipeline 
# main_pipeline()

### 4.3. Parameter Efficient Fine-Tuning (PEFT) 
We also explored PEFT methods to reduce computational cost while aiming to maintain performance.

- LoRA (Low-Rank Adaptation): This method freezes the pre-trained model weights and injects trainable low-rank matrices into the layers of the Transformer model.
- IA3 (Infused Adapter by Inhibiting and Amplifying Inner Activations): IA3 works by rescaling model activations with learned vectors, offering another way to adapt the model with fewer trainable parameters.

(Results for PEFT will be presented in the Results section below.)


### 4.4. Prompting Language Models
We evaluated the performance of several pre-trained Large Language Models (LLMs) using different prompting strategies on a subset of the test data (800 samples).
* **Models Tested:** Llama 3-8B, Mistral-7B-instruct-v0.3, Gemini 2.0 Flash.
* **Prompting Styles:**
    * **Zero-shot:** The model classifies text based only on class names and a general instruction.
        * *Example Instruction:* <br>
        "The classification must be one of the following... <br>
    * **Few-shot:** The model is provided with class names and a few examples of classified text.<br>
        * *Example Instruction:* <br>
        "The classification must be one of the following... <br>
        Examples..."<br>
    * **Few-shot + Chain of Thought (CoT):** The model is given class names, examples, and the reasoning behind the classification for those examples.
        * *Example Instruction:* <br>
        "Classify text as polite, somewhat polite, neutral, or impolite.<br>
         First, briefly explain your reasoning... <br>
         Examples: 'Happy to accommodate...' -> Polite. <br>
         Reasoning: ... <br>
         Classification: ..."<br>


### 6. Results and Discussion

This section presents the outcomes of the various experiments conducted, including hyperparameter tuning, the performance of fine-tuned models with and without domain adaptation, parameter-efficient fine-tuning techniques, and prompting large language models.

#### 6.1. Hyperparameter Tuning Outcomes

Hyperparameter tuning was performed for both BERT and RoBERTa models using a grid search approach. Key parameters tuned included learning rate and weight decay. The best configurations were selected based on the F1-score on the validation set and these models were subsequently trained for 10 epochs.

The F1 scores for various runs during this tuning phase are visualized below, comparing `bert-base-uncased` and `roberta-base`.

<img src="../assign2/llm_classification/results/fine_tunning_scores/hyperparameter_tuning.png" alt="F1 Scores of Different Model Runs during Hyperparameter Tuning" width="700"/>

*Caption: F1 Scores of different model configurations for BERT and RoBERTa during hyperparameter tuning.*

Based on the tuning process, the following hyperparameters were selected for the 10-epoch training runs:

  * **BERT (bert-base-uncased):** Learning Rate = 2e-5, Weight Decay = 0.01 
  * **RoBERTa (roberta-base):** Learning Rate = 2e-5, Weight Decay = 0.1 

#### 6.2. Fine-tuned Models with and without Domain Adaptation (MLM)

Domain adaptation via Masked Language Modeling (MLM) was applied to the RoBERTa model to adapt it to the specific linguistic nuances of the Polite Guard dataset before the final classification fine-tuning.

The **RoBERTa model with domain adaptation** emerged as the best-performing model:

#### 6.3. Parameter Efficient Fine-Tuning (PEFT) Results

We investigated two Parameter Efficient Fine-Tuning (PEFT) methods, LoRA (Low-Rank Adaptation) and IA3 (Infused Adapter by Inhibiting and Amplifying Inner Activations), to evaluate their effectiveness in reducing training time and computational resources while preserving performance compared to full fine-tuning.

| Model   | Method | ΔF1 (vs Base) | ΔTime (vs Base) |
| :------ | :----- | :------------ | :-------------- |
| BERT    | LORA   | -2.25%        | -26.16%         |
| BERT    | IA3    | -15.82%       | -26.03%         |
| RoBERTa | LORA   | -1.91%        | -25.33%         |
| RoBERTa | IA3    | -7.35%        | -24.42%         |
*Table Caption: Comparison of PEFT methods (LoRA and IA3) against their fully fine-tuned baselines for BERT and RoBERTa, showing the percentage change in F1-score and training time.*

**Key Observations from PEFT:**

  * Both LoRA and IA3 significantly reduced training time by approximately 25%
  * **LoRA** offered a much better trade-off, with a minimal drop in F1-score (around -2%) compared to its respective baseline.
  * **IA3** resulted in a more substantial decrease in performance, particularly for BERT.

The following plot illustrates the trade-off between F1-score and training time for the different models and training strategies.

<img src="../assign2/llm_classification/results/fine_tunning_scores/training_time_vs_f1_scatter_padded.png" alt="Model Performance F1 vs Training Time" width="700"/>

*Caption: F1 Score vs. Training Time (seconds) for different models and training methods, including full fine-tuning, domain adaptation (DA), LoRA, and IA3. RoBERTa+DA is the top-performing model in terms of F1-score, while PEFT methods show reduced training times.*

#### 6.4. LLM Prompting Results

Several Large Language Models (LLMs) were evaluated on a subset of 800 samples from the test set using different prompting strategies: zero-shot, few-shot, and few-shot with Chain-of-Thought (FS-CoT). The models tested included Llama 3-8B, Mistral-7B-instruct-v0.3, and Gemini 2.0 Flash.

<img src="../assign2/llm_classification/results/fine_tunning_scores/model_comparison_plot.png" alt="LLM Comparison on Intel/polite-guard" width="700"/>

*Caption: Weighted F1 scores for different LLMs (Gemini 2.0 Flash, Llama3-8B, Mistral-7B-v0.3) and prompt types (Zero-Shot, Few-Shot, FS-CoT) on an 800-sample subset of the Intel/polite-guard test data.*

**Key Findings from LLM Prompting:**

  * The performance of LLMs, with F1-scores generally ranging between \~60-70%, was significantly lower than that of the fine-tuned transformer models (which achieved over 92% F1).
  * **Few-shot Chain-of-Thought (FS-CoT)** prompting generally outperformed zero-shot and standard few-shot approaches across the tested models. This suggests that providing reasoning examples helps LLMs in this classification task, though it doesn't close the performance gap with fine-tuned models.
  * Surprisingly, **Gemini 2.0 Flash** performed worse than the smaller Llama 3-8B and Mistral-7B models in these experiments.
  * **Hypothesis:** The synthetic, LLM-generated nature of the Polite Guard dataset might create ambiguous class boundaries. Fine-tuned models, being trained extensively on this specific data distribution, may be better at capturing these nuanced (and potentially artificial) patterns than general-purpose LLMs operating with limited examples.

